In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:
fileVec = [
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_dji9443/xy264_z120_cml_smag017/forcetorque/force0/forcetorque0000.txt",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_dji9443/xy264_z120_cml_smag017_neq/forcetorque/force0/forcetorque0000.txt",
]

startStep = 1000
avgFactor = 0.6

labelVec = [
    r"Present LBM Cumulant $C_{smag}=0.17$ VERY coarse",
    r"Present LBM Cumulant $C_{smag}=0.17$ neq VERY coarse",
]

dxVec = [
    0.0006,
    0.0006,
]

rPhy = 0.12 # m

omegaRPMVec = [
    4800,
    4800,
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    "#A17808",
    '#FF5733',
    "#119100",
    "#B700FF",
    "#FA7BA1",
]


rho0LB = 1.0
gamma = 1.0
rhoPhy = 1.204
unitConverterVec = [ uc(dx, rhoPhy, gamma) for dx in dxVec ]

rLBVec = [rPhy / dx for dx in dxVec]
omegaLBVec = []
for case in range(0, len(fileVec)):
    omegaPhy = omegaRPMVec[case] * 2 * np.pi / 60
    omegaLBVec.append(unitConverterVec[case].phys_to_lb_omega(omegaPhy))

print(omegaLBVec)

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

In [ ]:

stepVec = []
forceVec = [[], [], []]
torqueVec = [[], [], []]
# refFxVec = []
for case in range(0, len(fileVec)):
    stepCol = read_col_probe(fileVec[case], 0, 2)
    fx = read_col_probe(fileVec[case], 2, 2)
    fy = read_col_probe(fileVec[case], 3, 2)
    fz = read_col_probe(fileVec[case], 4, 2)
    tx = read_col_probe(fileVec[case], 5, 2)
    ty = read_col_probe(fileVec[case], 6, 2)
    tz = read_col_probe(fileVec[case], 7, 2)
    
    idxStart = int(np.abs(stepCol - startStep).argmin())
    stepVec.append(stepCol[idxStart:])
    forceVec[0].append(fx[idxStart:])
    forceVec[1].append(fy[idxStart:])
    forceVec[2].append(fz[idxStart:])
    torqueVec[0].append(tx[idxStart:])
    torqueVec[1].append(ty[idxStart:])
    torqueVec[2].append(tz[idxStart:])


In [ ]:

figF, axF = plt.subplots(3, 2, figsize=(16, 8), facecolor='w', edgecolor='w', sharex=True)
for case in range(0, len(fileVec)):
    for i in [0, 1, 2]:
        axF[i][0].plot(stepVec[case], forceVec[i][case], c=colorVec[case])
        axF[i][1].plot(stepVec[case], torqueVec[i][case], c=colorVec[case])

axF[0][0].set_ylabel('Force-x')
axF[1][0].set_ylabel('Force-y')
axF[2][0].set_ylabel('Force-z')
axF[2][0].set_xlabel('Step')

axF[0][1].set_ylabel('Torque-x')
axF[1][1].set_ylabel('Torque-y')
axF[2][1].set_ylabel('Torque-z')
axF[2][1].set_xlabel('Step')

figF.tight_layout()

In [ ]:
figCt, axCt = plt.subplots(1, 1, figsize=(10, 6), facecolor='w', edgecolor='w')

coefFzVec = []
for case in range(0, len(fileVec)):
    cfzVec = []
    ctzVec = []
    area = np.pi * rLBVec[case]**2
    forceZCoef = forceVec[2][case] / (rho0LB * area * (omegaLBVec[case] * rLBVec[case])**2)
    coefFzVec.append(forceZCoef)
    
    idxStart = int(avgFactor * len(stepVec[case]))
    avgCT = np.mean(forceZCoef[idxStart:])
    avgStartStep = stepVec[case][idxStart]
    print(rf"Case {case}: mean thrust coefficient = {avgCT:.4f} (avg start step: {avgStartStep})")

for case in range(0, len(fileVec)):
    axCt.plot(stepVec[case], coefFzVec[case], c=colorVec[case], label=labelVec[case])

axCt.set_xlabel("Step")
axCt.set_ylabel(r"$C_T$")

axCt.legend(loc='upper right', frameon=False)
